# Day 20: Git, CI/CD and Docker
**Git, pipelines with security gates, AI pull request review, hardened images and the full stack in containers**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Setup
Git is installed in Colab. Docker is not, so Docker commands are for VS Code with Docker Desktop. The security scanners (bandit, pip-audit) run in both.

In [ ]:
# ===== Option A: Google Colab (active) =====
!git --version
!pip install -q pyyaml bandit pip-audit fastapi httpx

# ===== Option B: VS Code / local =====
# Install Git and Docker Desktop (see shared/Laptop_Setup_Guide.pdf), then:
# pip install pyyaml bandit pip-audit fastapi httpx
import os, shutil
shutil.rmtree("tasks-api", ignore_errors=True)
os.makedirs("tasks-api/app", exist_ok=True)
os.chdir("tasks-api")
print("Working in", os.getcwd())

## Setup and API key
With an `OPENAI_API_KEY` the notebook calls a real model. Without one, it runs in **offline demo mode** using a tiny stand-in, so every cell still works and you can study the code. Offline scores show how the code works, not real model quality.

The model name comes from the `LLM_MODEL` environment variable. The default is the program's approved default; confirm it against the approved model list in `docs/MODEL_GUIDE.md`.

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
usage() -> token counts of the last llm() call (estimated offline).
"""
import json, os, re
import numpy as np

def _get_key():
    if os.getenv("OFFLINE") == "1":                 # force offline demo mode (used by tools/check_notebooks.py)
        return None
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
# Default model: confirm against the approved model list (docs/MODEL_GUIDE.md). Override with LLM_MODEL.
MODEL = os.getenv("LLM_MODEL", "gpt-6.1-sol")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")
LAST_USAGE = {"input_tokens": 0, "output_tokens": 0}

_ENC = []

def count_tokens(text):
    if not _ENC:
        try:
            import tiktoken
            _ENC.append(tiktoken.get_encoding("o200k_base"))
        except Exception:
            _ENC.append(None)                        # no tiktoken or no download: use the estimate
    if _ENC[0] is not None:
        return len(_ENC[0].encode(text))
    return max(1, round(len(text) / 4))              # rough rule: about 4 characters per token

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "classify" in low:                            # SmartDesk ticket categories
        last = low.strip().splitlines()[-1]
        for label, words in [("billing", ["bill", "charge", "refund", "invoice", "payment"]),
                             ("network", ["signal", "network", "slow", "outage", "data", "5g"]),
                             ("account", ["password", "login", "account", "sim", "plan"])]:
            if any(w in last for w in words):
                return label
        return "other"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None, max_tokens=None):
    if ONLINE:
        kwargs = {"max_tokens": max_tokens} if max_tokens else {}
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}], **kwargs)
        LAST_USAGE.update(input_tokens=r.usage.prompt_tokens, output_tokens=r.usage.completion_tokens)
        return r.choices[0].message.content
    out = offline_llm(prompt, system)
    LAST_USAGE.update(input_tokens=count_tokens(system + prompt), output_tokens=count_tokens(out))
    return out

def usage():
    return dict(LAST_USAGE)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, usage, count_tokens, ONLINE, MODEL

## 2. A small app to version

In [ ]:
%%writefile app/main.py
from fastapi import FastAPI

app = FastAPI(title="Tasks API")
tasks = []

@app.get("/tasks")
def list_tasks():
    return tasks

In [ ]:
%%writefile requirements.txt
fastapi==0.115.0
uvicorn==0.30.6

## 3. First commits, a feature branch and a merge

In [ ]:
# ===== Option A: Google Colab (active) =====
!git init -q -b main
!git config user.name "Learner" && git config user.email "learner@example.com"
!git add . && git commit -q -m "Initial FastAPI app"
!git switch -q -c feature/health-check
!printf '\n@app.get("/health")\ndef health():\n    return {"status": "ok"}\n' >> app/main.py
!git add app/main.py && git commit -q -m "Add health check endpoint"
!git switch -q main && git merge -q feature/health-check && git branch -d feature/health-check
!git log --oneline

# ===== Option B: VS Code terminal =====
# Same commands without the "!" at the start.

## 4. A merge conflict, resolved

In [ ]:
!echo 'greeting = "Hello"' > greeting.py && git add . && git commit -q -m "Add greeting"
!git switch -q -c feature/japanese
!echo 'greeting = "Konnichiwa"' > greeting.py && git commit -q -am "Japanese greeting"
!git switch -q main
!echo 'greeting = "Namaste"' > greeting.py && git commit -q -am "Hindi greeting"
!git merge feature/japanese || true
!cat greeting.py

In [ ]:
# Resolve: keep both greetings, remove the markers
open("greeting.py", "w").write('greetings = ["Namaste", "Konnichiwa"]\n')
!git add greeting.py && git commit -q -m "Resolve greeting conflict: keep both"
!git log --oneline --graph | head -8

### TODO 1: Create a branch `feature/count`, add a `GET /tasks/count` endpoint, commit, and merge it into main.

In [ ]:
# TODO: write your code here

## 5. Push to Azure Repos (VS Code)
Create an empty repo in Azure DevOps first. Never paste tokens into a notebook.

In [ ]:
# ===== Option B: VS Code terminal (recommended) =====
# git remote add origin https://dev.azure.com/<org>/<project>/_git/tasks-api
# git push -u origin main
# git switch -c feature/priority-filter
# ...commit...
# git push -u origin feature/priority-filter
# Then open a pull request: Repos > Pull requests > New pull request
print("Remote commands are listed above")

## 6. NEW (v4): Security gates
We add a deliberately unsafe report feature, scan it with **bandit**, fail the gate, fix the code, and re-run. Never write code like `unsafe_report.py` in real work.

In [ ]:
%%writefile app/unsafe_report.py
import hashlib
import subprocess
import yaml

DB_PASSWORD = "admin123"                      # hard-coded password

def run_report(name):
    return subprocess.run(f"echo report {name}", shell=True, capture_output=True)   # shell injection risk

def load_settings(text):
    return yaml.load(text, Loader=yaml.Loader)   # unsafe loader

def ticket_hash(ticket_id):
    return hashlib.md5(str(ticket_id).encode()).hexdigest()   # weak hash

In [ ]:
import json, subprocess

BLOCK = {"MEDIUM", "HIGH"}

def run_bandit(path="app"):
    r = subprocess.run(["bandit", "-r", path, "-f", "json", "-q"], capture_output=True, text=True)
    data = json.loads(r.stdout or "{}")
    return [{"id": x["test_id"], "issue": x["issue_text"], "severity": x["issue_severity"],
             "file": x["filename"], "line": x["line_number"]} for x in data.get("results", [])]

def gate(findings, name="bandit"):
    bad = [f for f in findings if f["severity"] in BLOCK]
    status = "FAIL" if bad else "PASS"
    print(f"{name} gate: {status} ({len(bad)} blocking of {len(findings)} findings)")
    return 1 if bad else 0

run1 = run_bandit()
for f in run1:
    print(f"{f['severity']:<7} {f['id']}  {f['file']}:{f['line']}  {f['issue']}")
exit_code = gate(run1)

### Remediate
Each fix removes the risky pattern rather than hiding it with `# nosec`.

In [ ]:
%%writefile app/unsafe_report.py
import hashlib
import os
import subprocess
import yaml

DB_PASSWORD = os.getenv("DB_PASSWORD", "")      # from the environment or Key Vault, never in code

def run_report(name):
    return subprocess.run(["echo", "report", name], capture_output=True)   # list of args, no shell

def load_settings(text):
    return yaml.safe_load(text)                     # safe loader

def ticket_hash(ticket_id):
    return hashlib.sha256(str(ticket_id).encode()).hexdigest()

In [ ]:
run2 = run_bandit()
for f in run2:
    print(f"{f['severity']:<7} {f['id']}  {f['issue']}")
exit_code = gate(run2)

### Dependency scanning with pip-audit
`pip-audit` needs internet to query the vulnerability database, so the live cell is skipped in the offline check. The next cell parses a **sample** report in pip-audit's JSON shape so the gate logic still runs offline.

In [ ]:
%%writefile requirements-old.txt
requests==2.19.0

In [ ]:
# Live scan (needs internet)
!pip-audit -r requirements-old.txt --format json -o audit.json || true
!cat audit.json | head -c 600

In [ ]:
# SAMPLE output in pip-audit's JSON shape (for offline practice). CVE-2018-18074 affects requests before 2.20.0.
sample_audit = {"dependencies": [{"name": "requests", "version": "2.19.0", "vulns": [
    {"id": "PYSEC-2018-28", "aliases": ["CVE-2018-18074"], "fix_versions": ["2.20.0"],
     "description": "Authorization header sent on HTTPS to HTTP redirect."}]}]}

def audit_findings(report):
    return [{"package": d["name"], "version": d["version"], "id": v["id"], "fix": ", ".join(v["fix_versions"]),
             "severity": "MEDIUM"} for d in report["dependencies"] for v in d["vulns"]]

deps = audit_findings(sample_audit)
print(deps)
gate(deps, "pip-audit")

### Secret scanning (files and Git history)

In [ ]:
import re, pathlib
SECRET = re.compile(r"(?i)(api[_-]?key|secret|password|token)\s*=\s*['\"][^'\"]{6,}['\"]|sk-[A-Za-z0-9_-]{20,}")

def scan_secrets(root="."):
    hits = []
    for p in pathlib.Path(root).rglob("*.py"):
        for i, line in enumerate(p.read_text(errors="ignore").splitlines(), 1):
            if SECRET.search(line):
                hits.append(f"{p}:{i}  {line.strip()[:60]}")
    return hits

# Commit a fake key, then "delete" it
open("config.py", "w").write('API_KEY = "fake-key-for-training-123456"\n')
!git add config.py && git commit -q -m "Add config"
open("config.py", "w").write('import os\nAPI_KEY = os.getenv("API_KEY", "")\n')
!git commit -q -am "Read key from environment"

print("Files now:", scan_secrets() or "clean")
history = subprocess.run(["git", "log", "-p", "--all"], capture_output=True, text=True).stdout
print("Still in history:", bool(re.search(r"fake-key-for-training", history)))
print("Lesson: revoke and rotate the key first. Deleting the line does not remove it from history.")

### Vulnerability report v2
Build the report rows from the scan results. The full template is `shared/templates/Vulnerability_Report_Template.md`.

In [ ]:
import pandas as pd
rows = []
fixes = {"B602": "Pass a list of args, no shell", "B105": "Read from env / Key Vault",
         "B506": "yaml.safe_load", "B324": "Use sha256"}
reportable = [f for f in run1 if f["severity"] in BLOCK or f["id"] == "B105"]   # B105: low score, real risk
for n, f in enumerate(reportable, 1):
    rows.append({"ID": f"V-{n:02d}", "Finding": f["issue"][:45], "Tool": "bandit", "Severity": f["severity"],
                 "File": f"{os.path.basename(f['file'])}:{f['line']}", "Proposed fix": fixes.get(f["id"], "Review"),
                 "Fixed by": "Human", "Verified": "Yes" if not [x for x in run2 if x["id"] == f["id"]] else "No",
                 "Evidence": "bandit run 2"})
for d in deps:
    rows.append({"ID": f"V-{len(rows) + 1:02d}", "Finding": f"{d['package']} {d['version']} {d['id']}", "Tool": "pip-audit",
                 "Severity": d["severity"], "File": "requirements-old.txt", "Proposed fix": f"Upgrade to {d['fix']}",
                 "Fixed by": "Human", "Verified": "No", "Evidence": "re-run pip-audit after upgrade"})
rows.append({"ID": f"V-{len(rows) + 1:02d}", "Finding": "API key committed in config.py", "Tool": "secret scan",
             "Severity": "HIGH", "File": "config.py", "Proposed fix": "Revoke, rotate, env var", "Fixed by": "Human",
             "Verified": "Yes", "Evidence": "scan clean; key rotated"})
report = pd.DataFrame(rows)
print(report.to_string(index=False))
report.to_csv("vulnerability_report_v2.csv", index=False)      # copy rows into the Markdown template

## 7. NEW (v4): AI pull request review with Codex, human approval
In VS Code, paste the diff into Codex with the review prompt below. Here we show the same flow: get the diff, ask a reviewer (the LLM online, a rule-based stand-in offline), then a **human** decides.

In [ ]:
!git switch -q -c feature/delete-task
open("app/main.py", "a").write("""
@app.delete("/tasks/{idx}")
def delete_task(idx: int):
    tasks.pop(idx)
    return {"deleted": idx}
""")
!git add app/main.py && git commit -q -m "Add delete endpoint"
diff = subprocess.run(["git", "diff", "main", "--", "app/main.py"], capture_output=True, text=True).stdout
print(diff)

In [ ]:
REVIEW_PROMPT = """You are reviewing a pull request for a FastAPI service.
List up to 5 comments. For each: line, problem, suggested fix, severity (low, medium, high).
Check: bugs, missing tests, error handling, security. Do not approve or reject; a human decides.
Diff:
"""

def offline_review(diff):
    notes = []
    if ".pop(" in diff and "HTTPException" not in diff:
        notes.append("high: pop(idx) raises IndexError for a missing task; return 404 with HTTPException")
    if "def test_" not in diff:
        notes.append("medium: no test added for the new endpoint")
    if "@app.delete" in diff:
        notes.append("low: consider returning 204 No Content for deletes")
    return "\n".join(f"- {n}" for n in notes)

review = llm(REVIEW_PROMPT + diff, temperature=0) if ONLINE else offline_review(diff)
print("Reviewer comments:\n" + review)

decisions = {"404 for missing task": "accepted", "missing test": "accepted", "204 status": "rejected: keep 200 for now"}
print("\nHuman decisions:", decisions)
print("Merge approved by: human reviewer (branch policy requires 1 approval)")

## 8. A multi-stage pipeline with security gates

In [ ]:
%%writefile azure-pipelines.yml
trigger:
  - main

pr:
  - main

variables:
  imageName: tasks-api

stages:
  - stage: Test
    displayName: Lint and test
    jobs:
      - job: unit
        pool:
          vmImage: ubuntu-latest
        steps:
          - task: UsePythonVersion@0
            inputs:
              versionSpec: "3.12"
          - script: pip install -r requirements.txt pytest httpx ruff
            displayName: Install
          - script: ruff check . && python -m pytest -q --junitxml=results.xml
            displayName: Lint and tests
          - task: PublishTestResults@2
            condition: always()
            inputs:
              testResultsFiles: results.xml

  - stage: Security
    displayName: Security gates
    dependsOn: Test
    jobs:
      - job: scan
        pool:
          vmImage: ubuntu-latest
        steps:
          - script: pip install bandit pip-audit
            displayName: Install scanners
          - script: bandit -r app -ll
            displayName: SAST (fails on medium and high)
          - script: pip-audit -r requirements.txt
            displayName: Dependency audit
          - script: python tools/scan_secrets.py
            displayName: Secret scan

  - stage: Docker
    displayName: Build and scan image
    dependsOn: Security
    condition: and(succeeded(), eq(variables['Build.SourceBranch'], 'refs/heads/main'))
    jobs:
      - job: build
        pool:
          vmImage: ubuntu-latest
        steps:
          - script: docker build -t $(imageName):$(Build.BuildId) .
            displayName: docker build
          - script: trivy image --exit-code 1 --severity HIGH,CRITICAL $(imageName):$(Build.BuildId)
            displayName: Image scan (needs Trivy on the agent)

In [ ]:
import yaml
p = yaml.safe_load(open("azure-pipelines.yml"))
for st in p["stages"]:
    steps = [s.get("displayName") or s.get("task") for j in st["jobs"] for s in j["steps"]]
    print(f"{st['stage']:<9} after {st.get('dependsOn', '-'):<9} steps={steps}")

## 9. NEW (v4): Docker images, plain and hardened

In [ ]:
%%writefile Dockerfile.plain
FROM python:3.12
WORKDIR /app
COPY . .
RUN pip install -r requirements.txt
ENV API_KEY=changeme
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

In [ ]:
%%writefile Dockerfile
FROM python:3.12-slim
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app/ ./app
RUN useradd -m -u 10001 appuser
USER appuser
EXPOSE 8000
HEALTHCHECK CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

In [ ]:
%%writefile .dockerignore
.git
__pycache__/
*.pyc
.venv/
.env
tests/
*.md

In [ ]:
def lint_dockerfile(path):
    lines = [l.strip() for l in open(path) if l.strip() and not l.startswith("#")]
    text = "\n".join(lines)
    checks = {
        "slim or pinned small base": any(l.startswith("FROM") and ("slim" in l or "alpine" in l or "distroless" in l) for l in lines),
        "runs as non-root user": any(l.startswith("USER") and "root" not in l for l in lines),
        "no secrets in ENV": not re.search(r"ENV .*(KEY|SECRET|PASSWORD)=", text, re.I),
        "copies requirements before code": text.find("requirements.txt") < text.find("COPY app"),
        "no pip cache": "--no-cache-dir" in text,
        "has HEALTHCHECK": "HEALTHCHECK" in text,
    }
    return checks

for f in ["Dockerfile.plain", "Dockerfile"]:
    c = lint_dockerfile(f)
    print(f"{f:<17} {sum(c.values())}/{len(c)} checks passed")
    for name, ok in c.items():
        if not ok:
            print("   missing:", name)

In [ ]:
# ===== Option B: VS Code with Docker Desktop =====
# docker build -f Dockerfile.plain -t tasks-api:1.0 .
# docker build -t tasks-api:1.1 .
# docker images | grep tasks-api              # compare sizes
# docker run --rm tasks-api:1.1 id            # uid=10001(appuser), not root
# docker run -d -p 8000:8000 --name tasks tasks-api:1.1 && curl localhost:8000/health
# trivy image tasks-api:1.1                   # or: docker scout cves tasks-api:1.1
print("Run the Docker commands above in your terminal")

## 10. Containerising back end and front end
A cloud-ready API reads config from the environment and exposes `/health` and `/ready`. The React app uses a multi-stage build. Compose runs all three services.

In [ ]:
%%writefile app/cloud.py
import logging, os, sys
from fastapi import FastAPI
from fastapi.responses import JSONResponse

logging.basicConfig(stream=sys.stdout, level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("tasks-api")
app = FastAPI(title="Tasks API")

def database_reachable() -> bool:
    return os.getenv("SIMULATE_DB_DOWN") != "1"       # real apps ping the database here

@app.get("/health")
def health():
    return {"status": "ok", "env": os.getenv("APP_ENV", "local"), "version": os.getenv("APP_VERSION", "dev")}

@app.get("/ready")
def ready():
    if not database_reachable():
        log.warning("Readiness failed: database unreachable")
        return JSONResponse({"ready": False}, status_code=503)
    return {"ready": True}

In [ ]:
import importlib, sys
from fastapi.testclient import TestClient
sys.path.insert(0, os.getcwd())

def fresh_client(**env):
    for k in ["APP_ENV", "APP_VERSION", "SIMULATE_DB_DOWN"]:
        os.environ.pop(k, None)
    os.environ.update(env)
    import app.cloud as m
    importlib.reload(m)
    return TestClient(m.app)

print("Local:", fresh_client().get("/health").json())
print("Prod: ", fresh_client(APP_ENV="production", APP_VERSION="1.4.2").get("/health").json())
c = fresh_client(SIMULATE_DB_DOWN="1")
print("Ready with DB down:", c.get("/ready").status_code, c.get("/ready").json())

In [ ]:
os.makedirs("web", exist_ok=True)
open("web/Dockerfile", "w").write("""FROM node:22-slim AS build
WORKDIR /web
COPY package*.json ./
RUN npm ci
COPY . .
RUN npm run build

FROM nginx:alpine
COPY --from=build /web/dist /usr/share/nginx/html
EXPOSE 80
""")
open("docker-compose.yml", "w").write("""services:
  api:
    build: .
    ports: ["8000:8000"]
    environment:
      APP_ENV: local-compose
      MONGO_URI: mongodb://db:27017
    depends_on: [db]
  web:
    build: ./web
    ports: ["8080:80"]
    depends_on: [api]
  db:
    image: mongo:7
    volumes: ["mongo-data:/data/db"]
volumes:
  mongo-data:
""")
compose = yaml.safe_load(open("docker-compose.yml"))
for name, svc in compose["services"].items():
    print(f"{name:<4} image={svc.get('image', 'built from ' + str(svc.get('build'))):<24} ports={svc.get('ports', [])}")
# ===== Option B: VS Code with Docker Desktop =====
# docker compose up --build    then open http://localhost:8080 and http://localhost:8000/health

### TODO 2 (SmartDesk step 17): add a `Security` stage for SmartDesk with bandit, pip-audit, `npm audit --audit-level=high` and the secret scan. Record before and after scans in vulnerability report v2.

In [ ]:
# TODO: write your code here

## 11. Check yourself
1. Which bandit findings were HIGH, and how was each fixed?
2. Why is the deleted key still a problem?
3. Which Dockerfile checks did the plain image fail?

---
**Ganbatte (頑張って): *Do your best***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)